---
title: Loading data from the Open Data Cube
short_title: Loading data
subject: Beginner Guide
subtitle: Loading satellite imagery from the datacube with dc.load.
description: Loading satellite imagery from the datacube with dc.load.
keywords:
  - open-data-cube
  - odc
  - beginner-guide
---
---
This notebook introduces `dc.load`, the function that retrieves satellite imagery from the datacube.
It builds a query step by step, examines the returned data, and then adapts the query to load a different area.[^edits]

[^edits]: Tutorial notebooks update automatically, so edits may be overwritten during an update.
    Keep a working copy in a separate file to preserve changes.

## A. Objectives

- Load data with `dc.load`
- Read the returned `xarray.Dataset`
- Find ODC datasets before loading pixel data
- Load a selected set of ODC datasets

## B. Loading data

### 1. Connecting to the datacube

Every operation in this notebook starts with a connection to the datacube.
The connection is stored in `dc`, the conventional short name for a `Datacube` object, although another variable name would work just as well.

In [ ]:
from datacube import Datacube

dc = Datacube(app="loading_data")

### 2. Building a query for `dc.load`

A query gathers the choices needed to locate and load pixel data: the product, area, time, measurements, and output grid.
Keeping these choices in one Python dictionary makes them easy to inspect and change together.
Each key below has the same name as an argument accepted by `dc.load`.

In [ ]:
query = {
    "product": "s2_geomad_annual",
    "x": (98.80, 98.90),
    "y": (2.65, 2.55),
    "time": "2024",
    "measurements": ["red", "green", "blue"],
    "output_crs": "EPSG:32647",
    "resolution": (-30, 30),
}

The query uses these parameters:

- **product**: the product to load from the datacube.
  `dc.list_products()` returns the full list of available products.
- **x**, **y**: longitude and latitude ranges in degrees, each given as a tuple of two values.
  The ranges need to intersect the product's spatial extent; otherwise, the query returns no data.
- **time**: a date or date range.
  It accepts a year (`"2024"`), a month (`"2024-01"`), a specific date (`"2024-01-15"`), or a tuple such as `("2024-01-01", "2024-06-30")`.
- **measurements**: a list of the product measurements to return.
  `dc.list_measurements()` shows the measurements provided by each product.
- **output_crs**: the coordinate reference system for the loaded grid, given here as the EPSG code `"EPSG:32647"` for UTM zone 47N.
  This parameter is required when a product has no default output CRS, and a local UTM zone is a common choice.
- **resolution**: the pixel size in the units of `output_crs`, given in `(y, x)` order.
  The y value is usually negative because array rows proceed from north to south.

Other parameters support less common loading requirements:

- **datasets**: a list of ODC `Dataset` objects returned by `dc.find_datasets()`.
  This parameter supports the find-then-load workflow in section E, where a known set of indexed datasets is loaded.
- **group_by**: the rule for grouping multiple observations from the same period.
  A common value is `"solar_day"`.
- **dask_chunks**: the chunk configuration for lazy, parallel loading with Dask.
  It is useful when a load is too large to hold in memory at once.
- **resampling**: the method used to calculate pixel values during reprojection or a resolution change.
  Common values are `"nearest"`, the default, `"bilinear"`, and `"cubic"`.

The `**query` expression expands the dictionary into named arguments for `dc.load`.
ODC uses the search settings to find matching indexed datasets, reads the requested measurements, and arranges the pixels on the requested output grid.
The result is an `xarray.Dataset`, a multidimensional Python object covered in [04_xarray_for_odc.ipynb](04_xarray_for_odc.ipynb).

In [ ]:
ds = dc.load(**query)
ds # ds is a common abbreviation for dataset

## C. Reading the returned Dataset

The displayed summary describes the structure of the returned `xarray.Dataset`.
Its main parts are also available as attributes of `ds`:

- **Dimensions**: the sizes of the data axes.
  This result has one timestep (`time: 1`), 369 rows (`y: 369`), and 372 columns (`x: 372`).
- **Coordinates**: the labels along those axes.
  `time` contains the timestamp, `y` and `x` contain projected coordinates in `output_crs`, and `spatial_ref` describes the coordinate reference system.
- **Data variables**: the requested measurements.
  Each variable contains an array of measured values for every returned time and pixel position (`y`, `x`).

Each measurement is an `xarray.DataArray` and can be selected by name.
The following expression selects the red measurement while retaining its coordinates and dimensions:

In [ ]:
ds.red

A plot provides a quick spatial check of the loaded values:

In [ ]:
ds.red.plot()

The axes show the projected x and y coordinates, while the colour scale represents values from the red measurement.

## D. Finding datasets

Sometimes the indexed data needs to be inspected before any pixel arrays are loaded.
`dc.find_datasets` searches the datacube index and returns a list of matching ODC `Dataset` objects without reading their pixels.
This supports checks such as counting matches, examining metadata, or selecting a specific subset for `dc.load`.

An ODC `Dataset` is a metadata record for one indexed data item, including where its measurements are stored and when and where it applies.
For the product used here, each matching record describes an annual-composite dataset.
These objects are different from the `xarray.Dataset` of pixel arrays returned by `dc.load`.

The search uses the product, area, and time constraints from a loading query.
Output-grid settings such as `output_crs` and `resolution`, and the `measurements` selection, are only needed when pixels are loaded.

In [ ]:
datasets = dc.find_datasets(
    product="s2_geomad_annual",
    x=(98.80, 98.90),
    y=(2.65, 2.55),
    time=("2020", "2024"),
)

len(datasets)

`len(datasets)` reports how many indexed datasets match the product, area, and time constraints.
The first object can then be inspected directly:

In [ ]:
datasets[0]

Its metadata includes file locations, the native CRS, measurement paths, and timestamps.
No measurement arrays have been read at this stage.

## E. Loading selected datasets

A list returned by `dc.find_datasets` can be passed directly to `dc.load`.
The `datasets` parameter makes `dc.load` use those exact indexed records instead of repeating the catalogue search.
The requested measurements and output grid are still supplied because they control which arrays are read and how the result is arranged.

The x and y ranges in `dc.find_datasets` selected records whose spatial extents intersected the search area.
Each matching record retained its complete spatial extent.
The following `dc.load` call supplies no spatial bounds, so ODC can load those full extents and require substantially more memory than the first load.
For a practical analysis, pass x and y bounds to `dc.load` to limit the area, or use `dask_chunks` when a larger load is intentional.

In [ ]:
ds = dc.load(
    datasets=datasets,
    measurements=["red", "green", "blue"],
    output_crs="EPSG:32647",
    resolution=(-30, 30),
)

ds

The result still contains the `time`, `y`, and `x` dimensions and the red, green, and blue data variables, but their sizes and coordinate values can differ from the first load.
The selected list contains annual composites from several years, so the result has more time values.
Its x and y sizes and coordinate values can also change because the full spatial extents are loaded.

## F. Geometry helpers

A spatial query needs horizontal and vertical bounds, which can be cumbersome to calculate from a place of interest.
One common alternative is to build a bounding box from a centre point and a radius.
The Open Data Cube community maintains `odc.geo`, a library that provides geometry tools for this task.

In [ ]:
from odc.geo.geom import point

lat, lon = -8.65, 115.20  # Denpasar, Bali
bbox = point(lon, lat, crs="EPSG:4326").buffer(0.2).boundingbox
bbox

The `point` factory receives longitude followed by latitude and returns a `Geometry` object.
Its `crs` argument states that these coordinates use `EPSG:4326`, also known as WGS84.
The `.buffer(0.2)` call creates a geometry that extends 0.2 degrees from the point in every direction, which is roughly 22 km near the equator.
The final `.boundingbox` operation takes the rectangular bounds of that buffered geometry and stores them in `bbox`.

The bounding box can be drawn on an interactive basemap with `.explore()`.
This provides a quick check that the calculated area surrounds the intended location before data is loaded.

In [ ]:
bbox.explore()

The `.left`, `.right`, `.bottom`, and `.top` attributes provide the four edges needed by the query.
`query.update` replaces the x range, y range, and output CRS while retaining the earlier product, time, measurements, and resolution.
Bali lies in UTM zone 50S, so `EPSG:32750` replaces the UTM zone used for the first area.

In [ ]:
query.update({
    "x": (bbox.left, bbox.right),
    "y": (bbox.bottom, bbox.top),
    "output_crs": "EPSG:32750",
})

ds = dc.load(**query)
ds

The loaded red, green, and blue measurements can be combined for a quick RGB visualisation:

In [ ]:
ds.to_array().isel(time=0).plot.imshow(vmin=0, vmax=3000)

The plot selects the first time coordinate and uses display limits of 0 and 3000 to make the image easier to interpret.
These limits affect only the visualisation, not the values stored in `ds`.

## G. Next steps

The next notebook examines the `xarray.Dataset` returned by `dc.load` in more detail, including its dimensions and coordinates, subset selection, and variable combinations.

Continue to [04_xarray_for_odc.ipynb](04_xarray_for_odc.ipynb).